## Transferring Data from Device manufacturer sites to Influxdb
For flexibility and easy of data manipulation, we transfer the data to influx db.
This will also enable us to have the data downloaded and stored in one place - giving us ownership incase our subscription with the device manufacturer expires.

### Prepare tools for the project
First install the packages and modules necessary to implement the steps of the project.

The documentation for the influxdb_client_3 library can be found [here](https://docs.influxdata.com/influxdb3/cloud-dedicated/reference/client-libraries/v3/python/)

In [1]:
#Install operational packages to help in 
#organizing and querrying the data 
import pandas as pd
import numpy as np
import requests 
import json

#Modules to help interact with influxdb
import os, time
from influxdb_client_3 import InfluxDBClient3, Point


Create a magic function to help in skipping cells so that they run only once. Some cells especially those that involve uplaoding data to the database, can be skipped so that the data is not duplicated when the cells are are run multiple times

In [2]:
from IPython.core.magic import register_cell_magic

@register_cell_magic
def skip(line, cell):
    return

### Set up influxdb
Prepare the database to receive data from the device manufacturer urls 

In [3]:
#add this to load and upadte the token file
%reload_ext dotenv
%dotenv

#Define some of the arguments required 
#to write the data to the database

token = os.environ.get("INFLUXDB_TOKEN")
org         =   "Air quality - Harrisburg and Philadelphia"
host        =   "https://us-east-1-1.aws.cloud2.influxdata.com"
database    =   "pilot-harrisburg-home"

client = InfluxDBClient3(host=host, token=token, org=org, database=database)

### Upload the data from Devices to influxdb
Make API calls to the device urls and send the data to the influxdb database. Since different manufacturers have different procedures for API calls, we do upload the data in different batches based on the manufacturer

1. QUANTAQ

The guidelines to make API calls to quantaq can be found [here](https://docs.quant-aq.com/software-apis-and-libraries/quantaq-cloud-api)

In [17]:
#Load the api key and device serial numbers from the envrionment variable 
%reload_ext dotenv
%dotenv
QUANTAQ_APIKEY = os.environ.get("QUANTAQ_APIKEY")
QUANT001_SN = os.environ.get("QUANT001_SN")
QUANT002_SN = os.environ.get("QUANT002_SN")
CLARITY_APIKEY = os.environ.get("CLARITY_APIKEY")
CLARITY_ORGID = os.environ.get("CLARITY_ORGID")


In [5]:
def get_quantaq_data(url_ending):
    """
    A function that takes a list of words that go immediately after
    after the "https://api.quant-aq.com" of the url and help to 
    get access to scrap data from quantaq

    Args: 
        url_list (list):    A list of strings to be added to the url 
    
    Returns:
        response_dict (dictionary): a dictionary with the requested data        
    """

    #Define the fist part of the url
    url_first_part = "https://api.quant-aq.com/"

    #Initialize a continieoulsy increasing string to be used to complete the url
    url_additional = ""

    #Complete the url with user input strings 
    for adds_on in url_ending:
        url_additional = url_additional+adds_on+"/"        
    complete_url = url_first_part+url_additional

    #Make the call of the response to be returned 
    response_json = requests.get(complete_url, auth=(QUANTAQ_APIKEY,""))

    #Convert to dictionary for easy manipulation in python
    response_dict = json.loads(response_json.content)

    return response_dict

In [6]:

#Define the base url to be used in the api call

#Get the json data to be saved to influxdb from the quantaq
data_meta_quant002_dict = get_quantaq_data(['v1','devices',QUANT002_SN,'data'])



#Extract the data part of the transformed data 
# for easy saving to the influxdb database
data_quant002_dict = data_meta_quant002_dict['data']

#Transform the data to a dataframe for 
# easy compatibility with influxdb classes
data_quant002_df = pd.DataFrame(data=data_quant002_dict)

#Set the UTC timestamp to be the index for direct querrying
data_quant002_df_index = data_quant002_df.set_index('timestamp')


#Define the table name where the data will be saved in influxsb
measurement_name_quantaq = "quantaq-pilot"

#Save the data to influxdb to allow querying and future manipulation
client.write(record=data_quant002_df_index, 
             data_frame_measurement_name = measurement_name_quantaq)

In [7]:
#Define the base url to be used in the api call

#Get the json data to be saved to influxdb from the quantaq
data_meta_quant002_dict = get_quantaq_data(['v1','devices',QUANT002_SN,'data'])

#Extract the data part of the transformed data 
# for easy saving to the influxdb database
data_quant002_dict = data_meta_quant002_dict['data']

#Transform the data to a dataframe for 
# easy compatibility with influxdb classes
data_quant002_df = pd.DataFrame(data=data_quant002_dict)

#Set the UTC timestamp to be the index for direct querrying
data_quant002_df_index = data_quant002_df.set_index('timestamp')


#Define the table name where the data will be saved in influxsb
measurement_name_quantaq = "quantaq-pilot"

#Save the data to influxdb to allow querying and future manipulation
client.write(record=data_quant002_df_index, 
             data_frame_measurement_name = measurement_name_quantaq)

#### 2. Clarity
The general guidelines for accessing clarity data through the api can be found [here](https://api-guide.clarity.io/); These guideline use examples based on POSTMAN - a webtool for obtaining data using APIs. Clarity has a python library for retrieving data found [here](https://github.com/a2gov/clarityio). Here we use the python library.

First install the clarity library

In [8]:
%%skip
pip install clarityio

Make initial connections with the clarity library


In [ ]:
import clarityio # Offers methods to connect to the clarity api
from io import StringIO  # Helps to read file as csv and transform to df
api_connection = clarityio.ClarityAPIConnection(api_key=CLARITY_APIKEY, org=CLARITY_ORGID)

Getting the data

In [33]:
#First define the data you want to retrieve 
request_body_test1 = {
    'allDatasources': True,
    'outputFrequency': 'hour',
    'format':       'csv-wide',
    'metricSelect': 'only pm2_5ConcMass24HourRollingMean' 
    }

#get the data susing the definition 
response_wide = api_connection.get_recent_measurements(data=request_body_test1)
df_wide = pd.read_csv(StringIO(response_wide), sep=",")
df_wide


,datasourceId,sourceId,sourceType,outputFrequency,startOfPeriod,endOfPeriod,locationLatitude,locationLongitude,pm2_5ConcMass24HourRollingMean.raw,pm2_5ConcMass24HourRollingMean.status,pm2_5ConcMass24HourRollingMean.value
0,DSCFB3568,A47HWW6V,CLARITY_NODE,hour,2026-01-07T14:00:00Z,2026-01-07T15:00:00Z,40.264301,-76.851220,74.37,calibrated-ready,37.18
1,DALCU7773,A93NM49L,CLARITY_NODE,hour,2026-01-07T14:00:00Z,2026-01-07T15:00:00Z,37.879027,-122.301929,77.02,calibrated-ready,35.21
2,DYITV0908,A44MFTF3,CLARITY_NODE,hour,2026-01-07T14:00:00Z,2026-01-07T15:00:00Z,37.879027,-122.301929,76.63,calibrated-ready,36.53
3,DSCFB3568,A47HWW6V,CLARITY_NODE,hour,2026-01-07T13:00:00Z,2026-01-07T14:00:00Z,40.264301,-76.851220,76.04,calibrated-ready,37.97
4,DALCU7773,A93NM49L,CLARITY_NODE,hour,2026-01-07T13:00:00Z,2026-01-07T14:00:00Z,37.879027,-122.301929,79.97,calibrated-ready,36.44
...,...,...,...,...,...,...,...,...,...,...,...
64,DALCU7773,A93NM49L,CLARITY_NODE,hour,2026-01-06T17:00:00Z,2026-01-06T18:00:00Z,37.879027,-122.301929,64.77,calibrated-ready,29.03
65,DYITV0908,A44MFTF3,CLARITY_NODE,hour,2026-01-06T17:00:00Z,2026-01-06T18:00:00Z,37.879027,-122.301929,63.60,calibrated-ready,29.89
66,DSCFB3568,A47HWW6V,CLARITY_NODE,hour,2026-01-06T16:00:00Z,2026-01-06T17:00:00Z,40.264301,-76.851220,62.05,calibrated-ready,30.84
67,DALCU7773,A93NM49L,CLARITY_NODE,hour,2026-01-06T16:00:00Z,2026-01-06T17:00:00Z,37.879027,-122.301929,63.84,calibrated-ready,28.66


In [35]:
datasources_response = api_connection.get_datasources()
datasources_response

{'subscribingOrg': 'communUEVL',
 'datasources': [{'datasourceId': 'DALCU7773',
   'orgAnnotations': {'group': None, 'name': 'hbg-test1', 'tags': []},
   'sourceType': 'CLARITY_NODE',
   'currentSourceId': 'A93NM49L',
   'currentSubscriptionId': 'S39D9UKTB'},
  {'datasourceId': 'DYITV0908',
   'orgAnnotations': {'group': None, 'name': 'hbg-test2', 'tags': []},
   'sourceType': 'CLARITY_NODE',
   'currentSourceId': 'A44MFTF3',
   'currentSubscriptionId': 'S3A18TA94'},
  {'datasourceId': 'DSCFB3568',
   'orgAnnotations': {'group': None, 'name': 'Allison Hill', 'tags': []},
   'sourceType': 'CLARITY_NODE',
   'currentSourceId': 'A47HWW6V',
   'currentSubscriptionId': 'SF68M3EHF'}]}

In [ ]:
def get_clarity_data(url_ending):
    """
    A function that takes a list of words that go immediately after
    after the "https://clarity-data-api.clarity.io" of the url and help to 
    get access to scrap data from quantaq

    Args: 
        url_list (list):    A list of strings to be added to the url 
    
    Returns:
        response_dict (dictionary): a dictionary with the requested data        
    """
    
    #Define the fist part of the url
    url_first_part = "https://clarity-data-api.clarity.io"

    #Initialize a continieoulsy increasing string to be used to complete the url
    url_additional = ""

    #Complete the url with user input strings 
    for adds_on in url_ending:
        url_additional = url_additional + "/" + adds_on   
        
    complete_url = url_first_part+url_additional

    #Define the headers argument to be used to get the data
    headers_clarity = {
        'X-API-key' : CLARITY_APIKEY,
        'Accept' : "application/json"
    }

    #Make the call of the response to be returned 
    response_json = requests.get(complete_url, headers=headers_clarity)

    #Convert the file to a dictionary for easy maniputlation python
    response_dict = json.loads(response_json.content)

    return response_dict

In [ ]:
#Test clarity

test_url_clarity = ['v2', 'devices', 'nodes?org=communUEVL']
test2_url_clarity = ['v2', 'recent-datasource-measurements-query']
test3_url_clarity = ['v2', 'report-requests']

get_clarity_data(test_url_clarity)


[{'nodeId': 'A43H6HKC',
  'model': 'Node-S Global',
  'sku': 'CLA02-00001',
  'pairedAccessoryModules': [],
  'lifeStage': {'stage': 'purchased', 'when': '2025-10-30T18:16:32.946Z'}},
 {'nodeId': 'A636FW6Q',
  'model': 'Node-S Global',
  'sku': 'CLA02-00001',
  'pairedAccessoryModules': [],
  'lifeStage': {'stage': 'purchased', 'when': '2025-10-30T18:16:33.168Z'}},
 {'nodeId': 'A63CMY3J',
  'model': 'Node-S Global',
  'sku': 'CLA02-00001',
  'pairedAccessoryModules': [],
  'lifeStage': {'stage': 'purchased', 'when': '2025-10-30T18:16:33.266Z'}},
 {'nodeId': 'A44MFTF3',
  'model': 'Node-S Global',
  'sku': 'CLA02-00001',
  'pairedAccessoryModules': [{'accessoryId': 'MF99PJYR',
    'sku': 'CLA-MGAS-A01'}],
  'lifeStage': {'stage': 'working', 'when': '2025-11-07T18:15:33.271Z'},
  'location': {'type': 'Point',
   'coordinates': [-122.30192899608234, 37.87902695401437]}},
 {'nodeId': 'A47HWW6V',
  'model': 'Node-S Global',
  'sku': 'CLA02-00001',
  'pairedAccessoryModules': [{'accessoryId'

In [108]:
headers_clarity = {
        'X-API-key' : CLARITY_APIKEY,
        'Accept' : "application/json"
    }
requests.get('https://clarity-data-api.clarity.io/v2/report-requests', headers=headers_clarity).content

b'{"message": "Resource not found. Please, check the URL for typos."}'